    # Block 2 · Teil B: RSA zum Anfassen

    Dieses Notebook rechnet RSA mit **winzigen Zahlen** vor. Es ist absichtlich unsicher und dient nur zum Verstehen.
    Echte Programme nutzen geprüfte Bibliotheken mit riesigen Zahlen (Teil C).

**So arbeitest du**

- Zelle anklicken, dann **Umschalt + Enter**: Die Zelle wird ausgeführt. Arbeite von oben nach unten.
- Deine Antworten schreibst du auf das **Aufgabenblatt**. Die Überschriften hier tragen dieselben Aufgabennummern.
- **Vermute zuerst:** Schreib deine Vermutung auf, *bevor* du die Zelle ausführst. Danach vergleichen.
- Werte, die du ändern darfst, stehen in GROSSBUCHSTABEN oben in der Zelle.
- Seltsame Fehlermeldung? Menü **Kernel → Kernel neu starten** (englisch: *Restart Kernel*), dann alle Zellen von oben erneut ausführen.
- Dieses Notebook speichert Änderungen nur in **diesem Browser**. Auf Schulrechnern können sie nach dem Abmelden weg sein. Wichtig ist, was auf deinem Aufgabenblatt steht.

## Vorbereitung
Führe diese Zelle einmal aus. Sie stellt eine Hilfsfunktion bereit.

In [ ]:
from math import gcd
import time

def ist_primzahl(z):
    if z < 2:
        return False
    t = 2
    while t * t <= z:
        if z % t == 0:
            return False
        t += 1
    return True

print("Bereit.")

## B1 Schlüssel erzeugen, ver- und entschlüsseln

**Schritt 1:** Bob wählt zwei Primzahlen `P` und `Q` und einen öffentlichen Exponenten `E`. Daraus berechnet er seine beiden Schlüssel.

In [ ]:
P = 61    # erste Primzahl
Q = 53    # zweite Primzahl (anders als P)
E = 17    # öffentlicher Exponent

assert ist_primzahl(P) and ist_primzahl(Q) and P != Q, "P und Q müssen zwei verschiedene Primzahlen sein."
n = P * Q
phi = (P - 1) * (Q - 1)
assert gcd(E, phi) == 1, f"E = {E} passt nicht zu phi = {phi}. Probiere z. B. 3, 5, 7, 11, 13 oder 17."
d = pow(E, -1, phi)    # das Gegenstück zu E, es gilt: E · d mod phi = 1

print(f"n   = P · Q         = {P} · {Q} = {n}")
print(f"phi = (P-1) · (Q-1) = {P-1} · {Q-1} = {phi}")
print(f"d   = {d}   (Probe: E · d mod phi = {E * d % phi})")
print()
print(f"Öffentlicher Schlüssel (E, n) = ({E}, {n})   → darf jeder kennen")
print(f"Privater Schlüssel     (d, n) = ({d}, {n})   → bleibt geheim")

**Schritt 2:** Rechnen kann RSA nur mit Zahlen. Jedes Zeichen wird durch seine Nummer in der Zeichentabelle (Unicode) ersetzt.

In [ ]:
NACHRICHT = "HALLO"

m = [ord(z) for z in NACHRICHT]
print(m)

**Schritt 3:** Alice verschlüsselt mit Bobs **öffentlichem** Schlüssel: c = m<sup>E</sup> mod n

**Vermute zuerst:** Die Nachricht enthält zweimal „L“. Ergeben die beiden L im Geheimtext dieselbe Zahl oder verschiedene Zahlen?

In [ ]:
c = [pow(x, E, n) for x in m]    # pow(x, E, n) rechnet: x hoch E, davon der Rest bei Division durch n

for z, x, y in zip(NACHRICHT, m, c):
    print(f"{z!r}: {x:>4} ^ {E} mod {n} = {y}")
print()
print("Geheimtext:", ", ".join(str(y) for y in c))

**Schritt 4:** Bob entschlüsselt mit seinem **privaten** Schlüssel: m = c<sup>d</sup> mod n

In [ ]:
zurueck = [pow(y, d, n) for y in c]

print("Zahlen:", zurueck)
print("Text:  ", "".join(chr(x) for x in zurueck))

➜ Trage öffentlichen Schlüssel, privaten Schlüssel und Geheimtext in **B1** ein. Beantworte dann **B2**.

## B3 Ja oder Nein?

Bob hat gefragt: „Soll ich den Server heute Nacht neu starten? Antworte mit J oder N.“
Alice verschlüsselt ihre Antwort mit Bobs öffentlichem Schlüssel (17, 3233). Du fängst **1877** ab.
Den privaten Schlüssel kennst du **nicht**.

Trage bei `VERSUCH` eine mögliche Antwort ein und führe die Zelle erneut aus. Du darfst auch mehrere Zeichen auf einmal probieren.

In [ ]:
ABGEFANGEN = 1877
VERSUCH = "?"     # ← hier eine mögliche Antwort eintragen

for z in VERSUCH:
    y = pow(ord(z), 17, 3233)    # nur Bobs öffentlicher Schlüssel wird benutzt
    print(f"{z!r} → {y}", "  ← passt!" if y == ABGEFANGEN else "")

➜ Beschreibe dein Vorgehen in **B3**.

## B4 Der Angriff

Du hast eine Nachricht an den Admin abgefangen. Du kennst nur den öffentlichen Schlüssel und den Geheimtext
(Kasten auf dem Aufgabenblatt). Die Werte sind hier schon eingetragen.

In [ ]:
n_fremd = 1022117
E_fremd = 17
geheim = [757786, 141670, 141670, 461668, 531870, 724325, 17211, 70717,
          1018342, 226993, 276070, 672744, 148991, 609162, 927225]

Die Funktion `zerlege` probiert der Reihe nach alle möglichen Teiler aus (Probedivision).

**Vermute zuerst:** Wie lange braucht der Rechner, um n = 1 022 117 zu zerlegen: Millisekunden, Sekunden oder Minuten?

In [ ]:
def zerlege(n):
    """Probedivision: probiert alle möglichen Teiler der Reihe nach."""
    if n % 2 == 0:
        return 2, n // 2
    t = 3
    while t * t <= n:
        if n % t == 0:
            return t, n // t
        t += 2
    return None

start = time.perf_counter()
p_fremd, q_fremd = zerlege(n_fremd)
dauer = time.perf_counter() - start
print(f"n = {p_fremd} · {q_fremd}   (gefunden in {dauer * 1000:.2f} ms)")

Mit den beiden Primzahlen rechnet der Angreifer genau das nach, was Bob in B1 Schritt 1 gerechnet hat.

In [ ]:
phi_fremd = (p_fremd - 1) * (q_fremd - 1)
d_fremd = pow(E_fremd, -1, phi_fremd)
print("privater Schlüssel d =", d_fremd)
print("Nachricht:", "".join(chr(pow(y, d_fremd, n_fremd)) for y in geheim))

➜ Beantworte **B4**.

## B5 Wie groß ist groß genug?

Die nächste Zelle erzeugt RSA-Zahlen n mit wachsender Länge (in Bit) und misst, wie lange die Probedivision zum Zerlegen braucht.
Ein Versuch wird nach `ZEITLIMIT` Sekunden abgebrochen. Die Zelle läuft etwa 10 bis 20 Sekunden; solange steht links `[*]`.

**Vermute zuerst:** n wird um 4 Bit länger. Wie viel länger dauert dann das Zerlegen: doppelt so lang, viermal so lang oder zehnmal so lang?

In [ ]:
import math
import random

ZEITLIMIT = 5.0     # Sekunden pro Versuch
START_BIT = 20
SCHRITT_BIT = 4

def zufalls_primzahl(bits):
    while True:
        z = random.randrange(2 ** (bits - 1), 2 ** bits) | 1
        if ist_primzahl(z):
            return z

def zerlege_mit_frist(n, frist):
    """Probedivision wie in B4, bricht aber ab, wenn die Frist abläuft (Ergebnis None)."""
    if n % 2 == 0:
        return 2
    t, schritte = 3, 0
    while t * t <= n:
        if n % t == 0:
            return t
        t += 2
        schritte += 1
        if schritte % 65536 == 0 and time.perf_counter() > frist:
            return None
    return n

print(f"{'Bit von n':>10} | {'n':>22} | {'Zeit':>12}")
print("-" * 52)
letzte_bit, letzte_zeit = None, None
bits = START_BIT
while True:
    p = zufalls_primzahl(bits // 2)
    q = zufalls_primzahl(bits // 2)
    while q == p:
        q = zufalls_primzahl(bits // 2)
    n_test = p * q
    start = time.perf_counter()
    teiler = zerlege_mit_frist(n_test, start + ZEITLIMIT)
    dauer = time.perf_counter() - start
    if teiler is None:
        print(f"{n_test.bit_length():>10} | {n_test:>22} | {'abgebrochen':>12}")
        break
    print(f"{n_test.bit_length():>10} | {n_test:>22} | {dauer:>10.3f} s")
    letzte_bit, letzte_zeit = n_test.bit_length(), dauer
    bits += SCHRITT_BIT

print("-" * 52)
print(f"Hochgerechnet von der letzten vollständigen Messung ({letzte_bit} Bit):")
for ziel in (512, 2048, 3072):
    # Aufwand wächst mit der Wurzel aus n: Faktor 2 pro 2 Bit
    sek_log10 = math.log10(max(letzte_zeit, 1e-6)) + (ziel - letzte_bit) / 2 * math.log10(2)
    jahre_log10 = sek_log10 - math.log10(3600 * 24 * 365)
    print(f"  {ziel:>4} Bit: etwa 10^{jahre_log10:.0f} Jahre")
print("Zum Vergleich: Das Universum ist etwa 1,4 · 10^10 Jahre alt.")
print("ACHTUNG: Die Hochrechnung gilt nur für die Probedivision. Echte Angreifer haben bessere Verfahren.")

➜ Notiere die letzten drei **vollständig gemessenen** Zeilen und beantworte **B5 a) bis c)**.

## Für Schnelle: F1 Knackt euch gegenseitig

**Teil 1 (verschlüsseln):** Wähle zwei Primzahlen zwischen 100 und 2000 und eine kurze Nachricht. Fülle die Lücken `___` aus.
Gib deinem Partner **nur** n, E und den Geheimtext.

In [ ]:
P2 = ___        # Primzahl zwischen 100 und 2000
Q2 = ___        # zweite Primzahl, anders als P2
E2 = 17
TEXT = "___"    # kurze Nachricht

assert ist_primzahl(P2) and ist_primzahl(Q2) and P2 != Q2, "P2 und Q2 müssen zwei verschiedene Primzahlen sein."
n2 = P2 * Q2
assert gcd(E2, (P2 - 1) * (Q2 - 1)) == 1, "E2 passt nicht. Probiere 3, 5, 7, 11 oder 13."
print("Für deinen Partner:")
print("n =", n2, "  E =", E2)
print("Geheimtext:", [pow(ord(z), E2, n2) for z in TEXT])

**Teil 2 (knacken):** Trage die Werte deines Partners ein und schreibe den Angriff selbst. Vorlage: die Zellen aus B4.

In [ ]:
n_partner = ___
E_partner = ___
geheim_partner = [___]

start = time.perf_counter()
# 1. n zerlegen

# 2. phi und den privaten Schlüssel d berechnen

# 3. entschlüsseln und ausgeben

print(f"Fertig nach {(time.perf_counter() - start) * 1000:.2f} ms")